# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.28.3.1 — FAST
## GR vs GVH-DCC Matched Initial Shape Evolution Audit

### Mission

Avant le passage au champ central inhomogène `.3.3.28.4`, isoler proprement la différence entre :

\[
\boxed{\text{GR}}
\qquad\text{et}\qquad
\boxed{\text{branche réduite GVH-DCC/EA de `.3.3.28.3`}}
\]

pour la même géométrie diagonale initiale.

Le benchmark compare :

\[
\boxed{
\Delta a_i(\tau),\qquad
\Delta\sigma^2(\tau),\qquad
\Delta\mathrm{shape}(\tau)
}
\]

sans introduire de données observationnelles ni de restauration SI.

### Provenance canonique

Upstream Colab exécuté `.3.3.28.3` :

`1a9775915a2f578d237dda5da6761c3d9ff5d5c5d47433fb215c017d728cf1dd`

Taille : `45432` octets.

Structure : `20 cellules = 10 code + 10 markdown`.

Le checkpoint amont a établi, dans la branche homogène diagonale à shift nul :

\[
\sigma_{\hat i\hat j}
=
\frac{\dot\beta_i}{N}\delta_{ij},
\]

\[
\mathcal O_C=\sigma^2,
\]

et :

\[
a_i(\tau)=a_{i0}\tau^{P_i},
\]

avec :

\[
\sum_iP_i=1,
\]

\[
\sum_iP_i^2
=
\frac13+
\frac{6+3c_{13}+9c_2}
{9\left(1-c_{13}+\zeta_C/2\right)}.
\]

### Verrou méthodologique

« mêmes conditions initiales » ne signifie pas imposer arbitrairement les mêmes vitesses anisotropes dans deux théories ayant des contraintes hamiltoniennes différentes.

Le protocole principal sera :

- même géométrie initiale \(a_i(\tau_0)\) ;
- même temps de normalisation \(\tau_0\) ;
- même expansion moyenne \(\dot\alpha(\tau_0)\) ;
- même direction de l'anisotropie ;
- amplitude du cisaillement déterminée par la contrainte propre à chaque théorie.

Un second test montrera explicitement quand un **full-rate match** devient off-shell.

In [1]:
from __future__ import annotations
import json, sys, math
from pathlib import Path
import numpy as np
import pandas as pd
import sympy as sp

UPSTREAM = {
    "version": "0.3.2.7.3.7.3.3.28.3",
    "sha256": "1a9775915a2f578d237dda5da6761c3d9ff5d5c5d47433fb215c017d728cf1dd",
    "size_bytes": 45432,
    "cells": 20,
    "code_cells": 10,
    "markdown_cells": 10,
    "local_audit_pass": True,
    "restricted_covariant_shear_gate_closed": True,
    "reduced_physical_diagonal_dynamics_gate_closed": True,
    "physical_diagonal_functional_form_EOM_derived": True,
    "general_nested_Dk_EOM_derived": False,
    "Mercury_GVH_precession_prediction_authorized": False,
    "new_GVH_physics_validated": False,
}

G2831_UPSTREAM_PROVENANCE_PASS = all([
    UPSTREAM["size_bytes"] == 45432,
    UPSTREAM["cells"] == 20,
    UPSTREAM["code_cells"] == 10,
    UPSTREAM["markdown_cells"] == 10,
    UPSTREAM["local_audit_pass"],
    UPSTREAM["restricted_covariant_shear_gate_closed"],
    UPSTREAM["reduced_physical_diagonal_dynamics_gate_closed"],
    UPSTREAM["physical_diagonal_functional_form_EOM_derived"],
    not UPSTREAM["general_nested_Dk_EOM_derived"],
    not UPSTREAM["Mercury_GVH_precession_prediction_authorized"],
    not UPSTREAM["new_GVH_physics_validated"],
])

G2831_BLOCKED_OPEN_REFS = ["O-14","O-26","O-27","O-29","O-31","O-32","O-34"]

assert G2831_UPSTREAM_PROVENANCE_PASS

print("Python =", sys.version.split()[0])
print("NumPy =", np.__version__)
print("SymPy =", sp.__version__)
print("G2831_UPSTREAM_PROVENANCE_PASS =", G2831_UPSTREAM_PROVENANCE_PASS)
print("UPSTREAM_SHA256 =", UPSTREAM["sha256"])
print("G2831_BLOCKED_OPEN_REFS =", G2831_BLOCKED_OPEN_REFS)

Python = 3.13.15
NumPy = 2.1.3
SymPy = 1.14.0
G2831_UPSTREAM_PROVENANCE_PASS = True
UPSTREAM_SHA256 = 1a9775915a2f578d237dda5da6761c3d9ff5d5c5d47433fb215c017d728cf1dd
G2831_BLOCKED_OPEN_REFS = ['O-14', 'O-26', 'O-27', 'O-29', 'O-31', 'O-32', 'O-34']


# Niveau 1 — GVH

## 1.1 Relations exactes GR ↔ GVH-DCC

Définissons :

\[
\boxed{
A_\sigma
=
1-c_{13}+\frac{\zeta_C}{2}
}
\]

et :

\[
\boxed{
B_\alpha
=
6+3c_{13}+9c_2.
}
\]

Pour la branche réduite de `.3.3.28.3` :

\[
\sum_iP_i=1,
\]

\[
\boxed{
K_{\rm shape}
:=
\sum_iP_i^2
=
\frac13+\frac{B_\alpha}{9A_\sigma}.
}
\]

En GR :

\[
A_\sigma=1,
\qquad
B_\alpha=6,
\]

donc :

\[
\boxed{
K_{\rm shape}^{\rm GR}=1.
}
\]

L'écart exact à GR est :

\[
\boxed{
\Delta K_{\rm shape}
=
\frac{B_\alpha}{9A_\sigma}-\frac23.
}
\]

Le premier objectif est de factoriser exactement cette différence.

In [2]:
c13,c2,zeta = sp.symbols("c13 c2 zeta_C", real=True)

A_sigma = sp.simplify(1-c13+zeta/2)
B_alpha = sp.simplify(6+3*c13+9*c2)

K_shape = sp.simplify(sp.Rational(1,3)+B_alpha/(9*A_sigma))
delta_K = sp.factor(K_shape-1)

degeneracy_combo = sp.simplify(3*c13+3*c2-zeta)
delta_K_expected = sp.simplify(degeneracy_combo/(3*A_sigma))

G2831_DELTA_K_EXACT_FACTOR_PASS = (
    sp.simplify(delta_K-delta_K_expected) == 0
)

degeneracy_equation = sp.factor(B_alpha-6*A_sigma)
G2831_GR_DEGENERACY_SURFACE_EXACT_PASS = (
    sp.simplify(degeneracy_equation-3*degeneracy_combo) == 0
)

G2831_GR_LIMIT_EXACT_PASS = all([
    sp.simplify(A_sigma.subs({c13:0,c2:0,zeta:0})-1) == 0,
    sp.simplify(B_alpha.subs({c13:0,c2:0,zeta:0})-6) == 0,
    sp.simplify(K_shape.subs({c13:0,c2:0,zeta:0})-1) == 0,
])

assert G2831_DELTA_K_EXACT_FACTOR_PASS
assert G2831_GR_DEGENERACY_SURFACE_EXACT_PASS
assert G2831_GR_LIMIT_EXACT_PASS

print("A_sigma =", A_sigma)
print("B_alpha =", B_alpha)
print("K_shape =", K_shape)
print("Delta K_shape =", delta_K)
print("B_alpha - 6 A_sigma =", degeneracy_equation)
print("GR-degeneracy surface: 3*c13 + 3*c2 - zeta_C = 0")
print("G2831_GR_LIMIT_EXACT_PASS =", G2831_GR_LIMIT_EXACT_PASS)
print("G2831_GR_DEGENERACY_SURFACE_EXACT_PASS =",
      G2831_GR_DEGENERACY_SURFACE_EXACT_PASS)

A_sigma = -c13 + zeta_C/2 + 1
B_alpha = 3*c13 + 9*c2 + 6
K_shape = (6*c2 + zeta_C + 6)/(3*(-2*c13 + zeta_C + 2))
Delta K_shape = -2*(3*c13 + 3*c2 - zeta_C)/(3*(2*c13 - zeta_C - 2))
B_alpha - 6 A_sigma = 3*(3*c13 + 3*c2 - zeta_C)
GR-degeneracy surface: 3*c13 + 3*c2 - zeta_C = 0
G2831_GR_LIMIT_EXACT_PASS = True
G2831_GR_DEGENERACY_SURFACE_EXACT_PASS = True


## 1.2 Pourquoi les vitesses initiales ne peuvent pas être toutes identiques génériquement

La solution commune possède :

\[
\dot\alpha=\frac{1}{3\tau}.
\]

Pour la branche axisymétrique :

\[
(q_1,q_2,q_3)=(2s,-s,-s),
\]

la GR impose :

\[
\boxed{
s_{\rm GR}=\frac13.
}
\]

Si l'on prend **les taux anisotropes GR** et qu'on les injecte dans la contrainte GVH-DCC :

\[
A_\sigma\sum_i\dot\beta_i^2
-
B_\alpha\dot\alpha^2=0,
\]

le résidu devient :

\[
\boxed{
\mathcal C_{\rm full-rate}
=
-\frac{
3c_{13}+3c_2-\zeta_C
}{
3\tau^2
}.
}
\]

Ainsi un full-rate match est simultanément on-shell seulement si :

\[
\boxed{
3c_{13}+3c_2-\zeta_C=0.
}
\]

C'est pourquoi le benchmark principal garde la même géométrie, la même expansion moyenne et la même direction anisotrope, mais laisse chaque théorie fixer la norme de son cisaillement.

In [3]:
tau = sp.symbols("tau", positive=True, real=True)

alpha_dot = sp.Rational(1,3)/tau
q_GR = [
    sp.Rational(2,3),
    -sp.Rational(1,3),
    -sp.Rational(1,3),
]
beta_dot_GR = [q/tau for q in q_GR]

full_rate_residual = sp.factor(
    A_sigma*sum(x*x for x in beta_dot_GR)
    - B_alpha*alpha_dot**2
)

full_rate_expected = sp.simplify(
    -degeneracy_combo/(3*tau**2)
)

G2831_FULL_RATE_RESIDUAL_EXACT_PASS = (
    sp.simplify(full_rate_residual-full_rate_expected) == 0
)
G2831_GENERIC_FULL_RATE_MATCH_ONSHELL = False
G2831_DEGENERACY_SURFACE_FULL_RATE_MATCH_ONSHELL = True

assert G2831_FULL_RATE_RESIDUAL_EXACT_PASS
assert not G2831_GENERIC_FULL_RATE_MATCH_ONSHELL
assert G2831_DEGENERACY_SURFACE_FULL_RATE_MATCH_ONSHELL

print("full-rate GR->GVH Hamiltonian residual =", full_rate_residual)
print("G2831_FULL_RATE_RESIDUAL_EXACT_PASS =", G2831_FULL_RATE_RESIDUAL_EXACT_PASS)
print("G2831_GENERIC_FULL_RATE_MATCH_ONSHELL =", G2831_GENERIC_FULL_RATE_MATCH_ONSHELL)
print("G2831_DEGENERACY_SURFACE_FULL_RATE_MATCH_ONSHELL =",
      G2831_DEGENERACY_SURFACE_FULL_RATE_MATCH_ONSHELL)

full-rate GR->GVH Hamiltonian residual = -(3*c13 + 3*c2 - zeta_C)/(3*tau**2)
G2831_FULL_RATE_RESIDUAL_EXACT_PASS = True
G2831_GENERIC_FULL_RATE_MATCH_ONSHELL = False
G2831_DEGENERACY_SURFACE_FULL_RATE_MATCH_ONSHELL = True


## 1.3 Branche axisymétrique on-shell

Posons :

\[
(q_1,q_2,q_3)=(2s,-s,-s).
\]

Comme :

\[
\sum_iq_i^2=6s^2,
\]

la contrainte GVH-DCC donne :

\[
6A_\sigma s^2=\frac{B_\alpha}{9},
\]

donc :

\[
\boxed{
s_{\rm GVH}
=
\sqrt{
\frac{B_\alpha}{54A_\sigma}
}
}
\]

sur la branche réelle où :

\[
\frac{B_\alpha}{A_\sigma}>0.
\]

Les exposants deviennent :

\[
\boxed{
P^{\rm GVH}
=
\left(
\frac13+2s_{\rm GVH},
\frac13-s_{\rm GVH},
\frac13-s_{\rm GVH}
\right).
}
\]

En GR :

\[
\boxed{
P^{\rm GR}=(1,0,0).
}
\]

L'exposant discriminant de forme est :

\[
\boxed{
\nu_{\rm shape}
=
P_x-P_y
=
3s_{\rm GVH}
=
\sqrt{
\frac{B_\alpha}{6A_\sigma}
}.
}
\]

La GR correspond à :

\[
\nu_{\rm shape}^{\rm GR}=1.
\]

In [4]:
s = sp.sqrt(B_alpha/(54*A_sigma))
P_gvh = [
    sp.simplify(sp.Rational(1,3)+2*s),
    sp.simplify(sp.Rational(1,3)-s),
    sp.simplify(sp.Rational(1,3)-s),
]
P_gr = [sp.Integer(1),sp.Integer(0),sp.Integer(0)]

sumP_gvh = sp.simplify(sum(P_gvh))
sumP2_gvh = sp.simplify(sum(x*x for x in P_gvh))
nu_shape = sp.simplify(P_gvh[0]-P_gvh[1])
nu_shape_expected = sp.sqrt(B_alpha/(6*A_sigma))

G2831_AXISYMMETRIC_SUM_P_EXACT_PASS = (
    sp.simplify(sumP_gvh-1) == 0
)
G2831_AXISYMMETRIC_SUM_P2_EXACT_PASS = (
    sp.simplify(sumP2_gvh-K_shape) == 0
)
G2831_SHAPE_EXPONENT_EXACT_PASS = (
    sp.simplify(nu_shape-nu_shape_expected) == 0
)

assert G2831_AXISYMMETRIC_SUM_P_EXACT_PASS
assert G2831_AXISYMMETRIC_SUM_P2_EXACT_PASS
assert G2831_SHAPE_EXPONENT_EXACT_PASS

print("P_GVH =", P_gvh)
print("sum P_GVH =", sumP_gvh)
print("sum P_GVH^2 =", sumP2_gvh)
print("nu_shape =", nu_shape)
print("G2831_AXISYMMETRIC_SUM_P_EXACT_PASS =", G2831_AXISYMMETRIC_SUM_P_EXACT_PASS)
print("G2831_SHAPE_EXPONENT_EXACT_PASS =", G2831_SHAPE_EXPONENT_EXACT_PASS)

P_GVH = [2*sqrt((c13 + 3*c2 + 2)/(-2*c13 + zeta_C + 2))/3 + 1/3, 1/3 - sqrt((c13 + 3*c2 + 2)/(-2*c13 + zeta_C + 2))/3, 1/3 - sqrt((c13 + 3*c2 + 2)/(-2*c13 + zeta_C + 2))/3]
sum P_GVH = 1
sum P_GVH^2 = (-6*c2 - zeta_C - 6)/(3*(2*c13 - zeta_C - 2))
nu_shape = sqrt((c13 + 3*c2 + 2)/(-2*c13 + zeta_C + 2))
G2831_AXISYMMETRIC_SUM_P_EXACT_PASS = True
G2831_SHAPE_EXPONENT_EXACT_PASS = True


## 1.4 Observables de comparaison à géométrie initiale identique

Choisissons :

\[
\rho:=\frac{\tau}{\tau_0},
\qquad
a_i(\tau_0)=a_{i0}.
\]

Alors :

\[
\boxed{
a_i^{\rm GR}(\rho)
=
a_{i0}\rho^{P_i^{\rm GR}}
}
\]

et :

\[
\boxed{
a_i^{\rm GVH}(\rho)
=
a_{i0}\rho^{P_i^{\rm GVH}}.
}
\]

La différence directionnelle est :

\[
\boxed{
\Delta a_i
=
a_i^{\rm GVH}-a_i^{\rm GR}.
}
\]

Le cisaillement vaut :

\[
\boxed{
\sigma_{\rm GR}^2
=
\frac{2}{3\tau^2}
}
\]

et :

\[
\boxed{
\sigma_{\rm GVH}^2
=
\frac{B_\alpha}{9A_\sigma\tau^2}.
}
\]

Donc :

\[
\boxed{
\Delta\sigma^2
=
\frac{
3c_{13}+3c_2-\zeta_C
}{
3A_\sigma\tau^2
}.
}
\]

Pour la forme, utilisons :

\[
\mathcal R_{xy}:=\frac{a_x}{a_y}.
\]

Le ratio discriminant est :

\[
\boxed{
\frac{\mathcal R_{xy}^{\rm GVH}}
{\mathcal R_{xy}^{\rm GR}}
=
\rho^{\nu_{\rm shape}-1}.
}
\]

Enfin, comme :

\[
\sum_iP_i=1
\]

dans les deux branches :

\[
\boxed{
\frac{V_{\rm GVH}}{V_{\rm GR}}=1
}
\]

à normalisation initiale commune.

In [5]:
sigma2_GR = sp.simplify(sp.Rational(2,3)/tau**2)
sigma2_GVH = sp.simplify(B_alpha/(9*A_sigma*tau**2))
delta_sigma2 = sp.factor(sigma2_GVH-sigma2_GR)
delta_sigma2_expected = sp.simplify(
    degeneracy_combo/(3*A_sigma*tau**2)
)

G2831_DELTA_SIGMA2_EXACT_PASS = (
    sp.simplify(delta_sigma2-delta_sigma2_expected) == 0
)

rho = sp.symbols("rho", positive=True, real=True)
volume_GR = sp.simplify(rho**sum(P_gr))
volume_GVH = sp.simplify(rho**sumP_gvh)

G2831_VOLUME_EVOLUTION_MATCH_EXACT_PASS = (
    sp.simplify(volume_GVH/volume_GR-1) == 0
)

shape_ratio_relative = sp.simplify(
    rho**(nu_shape-1)
)

G2831_MATCHED_INITIAL_GEOMETRY_PROTOCOL_MATERIALIZED = True
G2831_PARAMETERIZED_SHAPE_DEVIATION_MATERIALIZED = True

assert G2831_DELTA_SIGMA2_EXACT_PASS
assert G2831_VOLUME_EVOLUTION_MATCH_EXACT_PASS

print("sigma_GR^2 =", sigma2_GR)
print("sigma_GVH^2 =", sigma2_GVH)
print("Delta sigma^2 =", delta_sigma2)
print("(Rxy_GVH/Rxy_GR) =", shape_ratio_relative)
print("V_GVH/V_GR =", sp.simplify(volume_GVH/volume_GR))
print("G2831_VOLUME_EVOLUTION_MATCH_EXACT_PASS =",
      G2831_VOLUME_EVOLUTION_MATCH_EXACT_PASS)

sigma_GR^2 = 2/(3*tau**2)
sigma_GVH^2 = 2*(c13 + 3*c2 + 2)/(3*tau**2*(-2*c13 + zeta_C + 2))
Delta sigma^2 = -2*(3*c13 + 3*c2 - zeta_C)/(3*tau**2*(2*c13 - zeta_C - 2))
(Rxy_GVH/Rxy_GR) = rho**(sqrt((c13 + 3*c2 + 2)/(-2*c13 + zeta_C + 2)) - 1)
V_GVH/V_GR = 1
G2831_VOLUME_EVOLUTION_MATCH_EXACT_PASS = True


# Niveau 2 — Benchmark numérique contrôlé

## 2.1 Même témoin interne que `.3.3.28.3`

On réutilise uniquement comme témoin de calcul :

\[
c_{13}=0.10,
\qquad
c_2=-0.02,
\qquad
\zeta_C=0.04.
\]

Ces valeurs ne sont pas présentées comme contraintes observationnelles.

Elles donnent :

\[
A_\sigma=0.92,
\qquad
B_\alpha=6.12.
\]

On compare les deux évolutions à :

\[
\rho=1,\ 2,\ 10,\ 100,
\]

avec :

\[
a_x(\tau_0)=a_y(\tau_0)=a_z(\tau_0)=1.
\]

Le volume doit rester identique dans les deux branches, tandis que la forme et le cisaillement peuvent diverger.

In [6]:
c13_num = 0.10
c2_num = -0.02
zeta_num = 0.04

A_num = float(A_sigma.subs({c13:c13_num,c2:c2_num,zeta:zeta_num}))
B_num = float(B_alpha.subs({c13:c13_num,c2:c2_num,zeta:zeta_num}))

s_num = math.sqrt(B_num/(54*A_num))
P_gvh_num = np.array([1/3+2*s_num,1/3-s_num,1/3-s_num], dtype=float)
P_gr_num = np.array([1.0,0.0,0.0])

rows = []
for rr in [1.0,2.0,10.0,100.0]:
    a_gr = rr**P_gr_num
    a_gvh = rr**P_gvh_num
    d = a_gvh-a_gr

    Rxy_gr = a_gr[0]/a_gr[1]
    Rxy_gvh = a_gvh[0]/a_gvh[1]
    shape_rel = Rxy_gvh/Rxy_gr

    V_gr = float(np.prod(a_gr))
    V_gvh = float(np.prod(a_gvh))

    # tau0=1, so tau=rho numerically for this dimensionless witness.
    sig_gr = 2/(3*rr**2)
    sig_gvh = B_num/(9*A_num*rr**2)

    rows.append({
        "rho": rr,
        "a_x_GR": a_gr[0],
        "a_x_GVH": a_gvh[0],
        "Delta_a_x": d[0],
        "a_y_GR": a_gr[1],
        "a_y_GVH": a_gvh[1],
        "Delta_a_y": d[1],
        "Rxy_GR": Rxy_gr,
        "Rxy_GVH": Rxy_gvh,
        "Rxy_GVH_over_GR": shape_rel,
        "V_GVH_over_GR": V_gvh/V_gr,
        "sigma2_GR": sig_gr,
        "sigma2_GVH": sig_gvh,
        "Delta_sigma2": sig_gvh-sig_gr,
    })

df_compare = pd.DataFrame(rows)

max_volume_ratio_error = float(np.max(np.abs(df_compare["V_GVH_over_GR"]-1.0)))
shape_ratio_10 = float(df_compare.loc[df_compare["rho"]==10.0,"Rxy_GVH_over_GR"].iloc[0])
shape_ratio_100 = float(df_compare.loc[df_compare["rho"]==100.0,"Rxy_GVH_over_GR"].iloc[0])

G2831_NUMERIC_VOLUME_MATCH_PASS = max_volume_ratio_error < 1e-13
G2831_NUMERIC_NONZERO_SHAPE_DEVIATION_WITNESS = abs(shape_ratio_10-1.0) > 1e-6
G2831_NUMERIC_NONZERO_SIGMA_DEVIATION_WITNESS = abs(
    float(df_compare.loc[df_compare["rho"]==10.0,"Delta_sigma2"].iloc[0])
) > 1e-10

G2831_NUMERIC_MATCHED_SHAPE_EVOLUTION_WITNESS_PASS = all([
    G2831_NUMERIC_VOLUME_MATCH_PASS,
    G2831_NUMERIC_NONZERO_SHAPE_DEVIATION_WITNESS,
    G2831_NUMERIC_NONZERO_SIGMA_DEVIATION_WITNESS,
])

assert G2831_NUMERIC_MATCHED_SHAPE_EVOLUTION_WITNESS_PASS

print("A_sigma =", A_num)
print("B_alpha =", B_num)
print("P_GR =", P_gr_num)
print("P_GVH =", P_gvh_num)
print(df_compare.to_string(index=False))
print("shape ratio GVH/GR at rho=10 =", shape_ratio_10)
print("shape ratio GVH/GR at rho=100 =", shape_ratio_100)
print("max |V_GVH/V_GR - 1| =", max_volume_ratio_error)
print("G2831_NUMERIC_MATCHED_SHAPE_EVOLUTION_WITNESS_PASS =",
      G2831_NUMERIC_MATCHED_SHAPE_EVOLUTION_WITNESS_PASS)

A_sigma = 0.92
B_alpha = 6.12
P_GR = [1. 0. 0.]
P_GVH = [ 1.03529745 -0.01764873 -0.01764873]
  rho  a_x_GR    a_x_GVH  Delta_a_x  a_y_GR  a_y_GVH  Delta_a_y  Rxy_GR    Rxy_GVH  Rxy_GVH_over_GR  V_GVH_over_GR  sigma2_GR  sigma2_GVH  Delta_sigma2
  1.0     1.0   1.000000   0.000000     1.0 1.000000   0.000000     1.0   1.000000         1.000000            1.0   0.666667    0.739130      0.072464
  2.0     2.0   2.049536   0.049536     1.0 0.987841  -0.012159     2.0   2.074762         1.037381            1.0   0.166667    0.184783      0.018116
 10.0    10.0  10.846696   0.846696     1.0 0.960177  -0.039823    10.0  11.296559         1.129656            1.0   0.006667    0.007391      0.000725
100.0   100.0 117.650805  17.650805     1.0 0.921940  -0.078060   100.0 127.612247         1.276122            1.0   0.000067    0.000074      0.000007
shape ratio GVH/GR at rho=10 = 1.1296559060752807
shape ratio GVH/GR at rho=100 = 1.2761224661307635
max |V_GVH/V_GR - 1| = 0.0
G2831_NUMERIC_MATC

## 2.2 Contrôle négatif : couplages non nuls mais évolution de forme identique à GR

Prenons :

\[
c_{13}=0.02,
\qquad
c_2=-0.01,
\qquad
\zeta_C=0.03.
\]

Alors :

\[
3c_{13}+3c_2-\zeta_C
=
0.
\]

Les couplages sont non nuls, mais :

\[
B_\alpha=6A_\sigma.
\]

Donc :

\[
\boxed{
K_{\rm shape}=1,
}
\]

\[
\boxed{
\nu_{\rm shape}=1,
}
\]

\[
\boxed{
\sigma_{\rm GVH}^2=\sigma_{\rm GR}^2.
}
\]

Le benchmark doit donc rendre exactement la même évolution de forme que GR.

Ce test empêche d'interpréter « couplages non nuls » comme « déviation nécessairement non nulle ».

In [7]:
c13_deg = 0.02
c2_deg = -0.01
zeta_deg = 0.03

A_deg = float(A_sigma.subs({c13:c13_deg,c2:c2_deg,zeta:zeta_deg}))
B_deg = float(B_alpha.subs({c13:c13_deg,c2:c2_deg,zeta:zeta_deg}))
combo_deg = 3*c13_deg+3*c2_deg-zeta_deg

s_deg = math.sqrt(B_deg/(54*A_deg))
P_deg = np.array([1/3+2*s_deg,1/3-s_deg,1/3-s_deg])

test_rhos = np.array([1.0,2.0,10.0,100.0])
max_a_deg_err = 0.0
max_shape_deg_err = 0.0
max_sigma_deg_err = 0.0

for rr in test_rhos:
    a_gr = rr**P_gr_num
    a_deg = rr**P_deg
    max_a_deg_err = max(max_a_deg_err,float(np.max(np.abs(a_deg-a_gr))))
    shape_gr = a_gr[0]/a_gr[1]
    shape_deg = a_deg[0]/a_deg[1]
    max_shape_deg_err = max(max_shape_deg_err,abs(shape_deg/shape_gr-1.0))
    sig_gr = 2/(3*rr**2)
    sig_deg = B_deg/(9*A_deg*rr**2)
    max_sigma_deg_err = max(max_sigma_deg_err,abs(sig_deg-sig_gr))

G2831_NONZERO_COUPLING_DEGENERACY_COMBO_ZERO_PASS = abs(combo_deg) < 1e-15
G2831_NONZERO_COUPLING_GR_SHAPE_DEGENERACY_WITNESS_PASS = all([
    max_a_deg_err < 1e-12,
    max_shape_deg_err < 1e-12,
    max_sigma_deg_err < 1e-12,
])

assert G2831_NONZERO_COUPLING_DEGENERACY_COMBO_ZERO_PASS
assert G2831_NONZERO_COUPLING_GR_SHAPE_DEGENERACY_WITNESS_PASS

print("A_deg =", A_deg)
print("B_deg =", B_deg)
print("3*c13+3*c2-zeta =", combo_deg)
print("P_degenerate =", P_deg)
print("max a_i error vs GR =", max_a_deg_err)
print("max shape ratio error vs GR =", max_shape_deg_err)
print("max sigma^2 error vs GR =", max_sigma_deg_err)
print("G2831_NONZERO_COUPLING_GR_SHAPE_DEGENERACY_WITNESS_PASS =",
      G2831_NONZERO_COUPLING_GR_SHAPE_DEGENERACY_WITNESS_PASS)

A_deg = 0.995
B_deg = 5.97
3*c13+3*c2-zeta = 0.0
P_degenerate = [ 1.00000000e+00 -5.55111512e-17 -5.55111512e-17]
max a_i error vs GR = 2.220446049250313e-16
max shape ratio error vs GR = 2.220446049250313e-16
max sigma^2 error vs GR = 8.673617379884035e-19
G2831_NONZERO_COUPLING_GR_SHAPE_DEGENERACY_WITNESS_PASS = True


# Niveau 3 — Sensibilité analytique près de GR

Le discriminateur exact de forme est :

\[
\boxed{
\epsilon_{\rm shape}
:=
\nu_{\rm shape}-1
=
\sqrt{
\frac{B_\alpha}{6A_\sigma}
}
-1.
}
\]

Pour des couplages petits, introduisons un paramètre comptable \(\varepsilon\) :

\[
c_{13}\rightarrow\varepsilon c_{13},
\qquad
c_2\rightarrow\varepsilon c_2,
\qquad
\zeta_C\rightarrow\varepsilon\zeta_C.
\]

Le développement au premier ordre donne :

\[
\boxed{
\epsilon_{\rm shape}
=
\frac{
3c_{13}+3c_2-\zeta_C
}{4}
+
O(c^2).
}
\]

Ainsi le même invariant linéaire :

\[
\boxed{
3c_{13}+3c_2-\zeta_C
}
\]

contrôle à la fois :

- l'écart de la contrainte de Kasner ;
- l'incompatibilité d'un full-rate match générique ;
- l'écart du cisaillement ;
- l'écart de forme au premier ordre.

Mais ce discriminateur n'est pas encore une signature propre au terme cubique : \(c_{13}\) et \(c_2\) appartiennent déjà au secteur Einstein-æther.

In [8]:
eps = sp.symbols("eps", real=True)

nu_eps = sp.sqrt(
    B_alpha.subs({
        c13:eps*c13,
        c2:eps*c2,
        zeta:eps*zeta,
    }) /
    (
        6*A_sigma.subs({
            c13:eps*c13,
            c2:eps*c2,
            zeta:eps*zeta,
        })
    )
)

epsilon_shape_series = sp.series(
    sp.simplify(nu_eps-1),
    eps,
    0,
    2
).removeO()

epsilon_shape_linear_coeff = sp.simplify(
    sp.diff(epsilon_shape_series,eps).subs(eps,0)
)

epsilon_shape_expected = sp.simplify(
    degeneracy_combo/4
)

G2831_SMALL_COUPLING_SHAPE_EXPANSION_PASS = (
    sp.simplify(epsilon_shape_linear_coeff-epsilon_shape_expected) == 0
)

# Random internal algebraic consistency sweep; not physical priors.
rng = np.random.default_rng(2831)
sweep_rows = []
max_closed_err = 0.0

for _ in range(200):
    c13v = float(rng.uniform(-0.15,0.15))
    c2v = float(rng.uniform(-0.08,0.08))
    zv = float(rng.uniform(-0.10,0.10))

    Av = 1-c13v+zv/2
    Bv = 6+3*c13v+9*c2v
    if Av <= 0 or Bv <= 0:
        continue

    Kv_direct = 1/3+Bv/(9*Av)
    Kv_closed = 1+(3*c13v+3*c2v-zv)/(3*Av)
    err = abs(Kv_direct-Kv_closed)
    max_closed_err = max(max_closed_err,err)

G2831_RANDOM_ALGEBRAIC_SWEEP_PASS = max_closed_err < 1e-13

assert G2831_SMALL_COUPLING_SHAPE_EXPANSION_PASS
assert G2831_RANDOM_ALGEBRAIC_SWEEP_PASS

print("epsilon_shape first-order coefficient =",
      epsilon_shape_linear_coeff)
print("expected =", epsilon_shape_expected)
print("random sweep max closed-form error =", max_closed_err)
print("G2831_SMALL_COUPLING_SHAPE_EXPANSION_PASS =",
      G2831_SMALL_COUPLING_SHAPE_EXPANSION_PASS)
print("G2831_RANDOM_ALGEBRAIC_SWEEP_PASS =",
      G2831_RANDOM_ALGEBRAIC_SWEEP_PASS)

epsilon_shape first-order coefficient = 3*c13/4 + 3*c2/4 - zeta_C/4
expected = 3*c13/4 + 3*c2/4 - zeta_C/4
random sweep max closed-form error = 2.220446049250313e-16
G2831_SMALL_COUPLING_SHAPE_EXPANSION_PASS = True
G2831_RANDOM_ALGEBRAIC_SWEEP_PASS = True


## 3.1 Audit Prescribed / Derived

`.3.3.28.3.1` ne dérive aucun nouveau couplage numérique.

Il établit une **carte paramétrique exacte** :

\[
(c_{13},c_2,\zeta_C)
\longrightarrow
\left(
P_i,
\sigma^2,
\mathcal R_{xy}
\right)
\]

dans la branche homogène réduite.

Ce qui est dérivé :

\[
\Delta K_{\rm shape},
\quad
\Delta\sigma^2,
\quad
\nu_{\rm shape},
\quad
\Delta a_i(\rho)
\]

pour des couplages donnés.

Ce qui reste prescrit ou non sélectionné :

\[
c_{13},c_2,\zeta_C,
\]

les conditions d'intégration individuelles,

et le mapping vers une source astrophysique localisée.

Le témoin :

\[
(0.10,-0.02,0.04)
\]

est uniquement un **numeric witness**.

Aucune déviation calculée ici ne doit être appelée « prédiction observationnelle GVH ».

In [9]:
pd_ledger = pd.DataFrame([
    {
        "object":"A_sigma, B_alpha",
        "status":"DERIVED_FROM_REDUCED_ACTION_COUPLINGS",
        "observation_ready":False,
    },
    {
        "object":"Delta K_shape",
        "status":"EXACT_PARAMETERIZED_DERIVED",
        "observation_ready":False,
    },
    {
        "object":"Delta sigma^2(tau)",
        "status":"EXACT_PARAMETERIZED_DERIVED",
        "observation_ready":False,
    },
    {
        "object":"nu_shape",
        "status":"EXACT_PARAMETERIZED_DERIVED",
        "observation_ready":False,
    },
    {
        "object":"Delta a_i(rho)",
        "status":"DERIVED_FOR_MATCHED_INITIAL_GEOMETRY_PROTOCOL",
        "observation_ready":False,
    },
    {
        "object":"3*c13+3*c2-zeta_C",
        "status":"EXACT_GR_DEGENERACY_COMBINATION",
        "observation_ready":False,
    },
    {
        "object":"c13,c2,zeta_C",
        "status":"NOT_DERIVED_NUMERICALLY_HERE",
        "observation_ready":False,
    },
    {
        "object":"localized D_phys(t,x)",
        "status":"NOT_YET_DERIVED",
        "observation_ready":False,
    },
    {
        "object":"Mercury perihelion correction",
        "status":"NOT_AUTHORIZED",
        "observation_ready":False,
    },
])

G2831_COUPLINGS_NUMERICALLY_DERIVED = False
G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE = False
G2831_INHOMOGENEOUS_FIELD_DERIVED = False
G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED = False
G2831_NEW_GVH_PHYSICS_VALIDATED = False
G2831_REAL_DATA_READY = False

assert not G2831_COUPLINGS_NUMERICALLY_DERIVED
assert not G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE
assert not G2831_INHOMOGENEOUS_FIELD_DERIVED
assert not G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED
assert not G2831_NEW_GVH_PHYSICS_VALIDATED
assert not G2831_REAL_DATA_READY

print(pd_ledger.to_string(index=False))
print("G2831_COUPLINGS_NUMERICALLY_DERIVED =", G2831_COUPLINGS_NUMERICALLY_DERIVED)
print("G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE =",
      G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE)
print("G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED =",
      G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED)

                       object                                        status  observation_ready
             A_sigma, B_alpha         DERIVED_FROM_REDUCED_ACTION_COUPLINGS              False
                Delta K_shape                   EXACT_PARAMETERIZED_DERIVED              False
           Delta sigma^2(tau)                   EXACT_PARAMETERIZED_DERIVED              False
                     nu_shape                   EXACT_PARAMETERIZED_DERIVED              False
               Delta a_i(rho) DERIVED_FOR_MATCHED_INITIAL_GEOMETRY_PROTOCOL              False
            3*c13+3*c2-zeta_C               EXACT_GR_DEGENERACY_COMBINATION              False
                c13,c2,zeta_C                  NOT_DERIVED_NUMERICALLY_HERE              False
        localized D_phys(t,x)                               NOT_YET_DERIVED              False
Mercury perihelion correction                                NOT_AUTHORIZED              False
G2831_COUPLINGS_NUMERICALLY_DERIVED = False
G2831_

# Niveau 4 — Verdict scientifique

`.3.3.28.3.1` ferme le benchmark **GR vs branche réduite GVH-DCC/EA** avec géométrie initiale contrôlée.

### Résultat A — récupération GR

\[
\boxed{
c_{13}=c_2=\zeta_C=0
\Longrightarrow
P=(1,0,0),
\quad
\sigma^2=\frac{2}{3\tau^2}.
}
\]

### Résultat B — déviation paramétrique exacte

\[
\boxed{
\Delta K_{\rm shape}
=
\frac{
3c_{13}+3c_2-\zeta_C
}{
3A_\sigma
}.
}
\]

\[
\boxed{
\Delta\sigma^2
=
\frac{
3c_{13}+3c_2-\zeta_C
}{
3A_\sigma\tau^2
}.
}
\]

\[
\boxed{
\frac{\mathcal R_{xy}^{\rm GVH}}
{\mathcal R_{xy}^{\rm GR}}
=
\rho^{
\sqrt{B_\alpha/(6A_\sigma)}-1
}.
}
\]

### Résultat C — surface exacte de dégénérescence GR

\[
\boxed{
3c_{13}+3c_2-\zeta_C=0
}
\]

implique une évolution de forme identique à GR dans cette branche, même avec des couplages non nuls.

### Résultat D — même volume, forme différente

À normalisation initiale identique :

\[
\boxed{
V_{\rm GVH}/V_{\rm GR}=1.
}
\]

La discrimination porte donc sur l'anisotropie et non sur la loi globale du volume dans ce benchmark.

### Verrou d'interprétation

Cette déviation n'est pas une signature propre du nouveau secteur cubique, car :

\[
c_{13},c_2
\]

contribuent déjà dans Einstein-æther et, sur le trièdre aligné :

\[
\zeta_C
\]

reste absorbable dans un décalage de coefficient de cisaillement.

Ainsi :

\[
\boxed{
\texttt{PARAMETERIZED\_SHAPE\_DEVIATION\_MATERIALIZED=True}
}
\]

mais :

\[
\boxed{
\texttt{NEW\_GVH\_PHYSICS\_VALIDATED=False}.
}
\]

Le passage au champ central inhomogène peut maintenant être ouvert sans ambiguïté sur le benchmark homogène.

In [10]:
LEVEL1 = "GR_LIMIT_AND_MATCHED_INITIAL_GEOMETRY_PROTOCOL_EXACTLY_CLASSIFIED"
LEVEL2 = "PARAMETERIZED_SHAPE_SHEAR_AND_AXIS_EVOLUTION_DIFFERENCES_MATERIALIZED"
LEVEL3 = "NUMERIC_NONDEGENERATE_AND_NONZERO_COUPLING_GR_DEGENERACY_WITNESSES_PASS"
LEVEL4 = "BLOCKED_UNIQUE_CUBIC_ATTRIBUTION_INHOMOGENEOUS_FIELD_AND_MERCURY_PREDICTION"

G2831_GR_VS_GVH_MATCHED_SHAPE_GATE_CLOSED = all([
    G2831_GR_LIMIT_EXACT_PASS,
    G2831_DELTA_K_EXACT_FACTOR_PASS,
    G2831_GR_DEGENERACY_SURFACE_EXACT_PASS,
    G2831_FULL_RATE_RESIDUAL_EXACT_PASS,
    G2831_AXISYMMETRIC_SUM_P_EXACT_PASS,
    G2831_AXISYMMETRIC_SUM_P2_EXACT_PASS,
    G2831_SHAPE_EXPONENT_EXACT_PASS,
    G2831_DELTA_SIGMA2_EXACT_PASS,
    G2831_VOLUME_EVOLUTION_MATCH_EXACT_PASS,
])

G2831_NUMERIC_BENCHMARK_GATE_CLOSED = all([
    G2831_NUMERIC_MATCHED_SHAPE_EVOLUTION_WITNESS_PASS,
    G2831_NONZERO_COUPLING_GR_SHAPE_DEGENERACY_WITNESS_PASS,
    G2831_SMALL_COUPLING_SHAPE_EXPANSION_PASS,
    G2831_RANDOM_ALGEBRAIC_SWEEP_PASS,
])

G2831_LOCAL_AUDIT_PASS = all([
    G2831_UPSTREAM_PROVENANCE_PASS,
    G2831_GR_VS_GVH_MATCHED_SHAPE_GATE_CLOSED,
    G2831_NUMERIC_BENCHMARK_GATE_CLOSED,
    G2831_MATCHED_INITIAL_GEOMETRY_PROTOCOL_MATERIALIZED,
    G2831_PARAMETERIZED_SHAPE_DEVIATION_MATERIALIZED,
    not G2831_GENERIC_FULL_RATE_MATCH_ONSHELL,
    not G2831_COUPLINGS_NUMERICALLY_DERIVED,
    not G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE,
    not G2831_INHOMOGENEOUS_FIELD_DERIVED,
    not G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED,
    not G2831_NEW_GVH_PHYSICS_VALIDATED,
    not G2831_REAL_DATA_READY,
])

G2831_OBSTRUCTIONS = [
    "MATCHED_FULL_INITIAL_RATES_ARE_GENERICALLY_OFFSHELL_BETWEEN_THE_TWO_THEORIES",
    "COUPLINGS_c13_c2_zetaC_ARE_NOT_NUMERICALLY_THEORY_SELECTED_HERE",
    "ALIGNED_BRANCH_DEVIATION_IS_NOT_UNIQUELY_ATTRIBUTABLE_TO_THE_CUBIC_OPERATOR",
    "GENERAL_INHOMOGENEOUS_D_phys_OF_t_x_NOT_DERIVED",
    "LOCALIZED_SOURCE_AND_NBODY_SPATIAL_MATCHING_NOT_MATERIALIZED",
    "MERCURY_GVH_PRECESSION_NOT_AUTHORIZED",
]

G2831_NEXT_AUTHORIZED = (
    "0.3.2.7.3.7.3.3.28.4_"
    "INHOMOGENEOUS_CENTRAL_FIELD_DIAGONAL_EMBEDDING_AND_NBODY_MATCHING_GATE_AUDIT"
)

assert G2831_LOCAL_AUDIT_PASS

artifact = {
    "notebook": (
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.3.1_"
        "GR_vs_GVH_DCC_Matched_Initial_Shape_Evolution_Audit_FAST"
    ),
    "upstream": UPSTREAM,
    "open_refs": G2831_BLOCKED_OPEN_REFS,
    "levels": {
        "LEVEL1": LEVEL1,
        "LEVEL2": LEVEL2,
        "LEVEL3": LEVEL3,
        "LEVEL4": LEVEL4,
    },
    "exact_relations": {
        "A_sigma": str(A_sigma),
        "B_alpha": str(B_alpha),
        "K_shape": str(K_shape),
        "Delta_K_shape": str(delta_K),
        "degeneracy_combination": str(degeneracy_combo),
        "full_rate_residual": str(full_rate_residual),
        "nu_shape": str(nu_shape),
        "Delta_sigma2": str(delta_sigma2),
    },
    "numeric_witness": {
        "couplings": {
            "c13": c13_num,
            "c2": c2_num,
            "zeta_C": zeta_num,
        },
        "A_sigma": A_num,
        "B_alpha": B_num,
        "P_GR": P_gr_num.tolist(),
        "P_GVH": P_gvh_num.tolist(),
        "shape_ratio_GVH_over_GR_rho10": shape_ratio_10,
        "shape_ratio_GVH_over_GR_rho100": shape_ratio_100,
        "max_volume_ratio_error": max_volume_ratio_error,
    },
    "nonzero_coupling_GR_degeneracy_witness": {
        "couplings": {
            "c13": c13_deg,
            "c2": c2_deg,
            "zeta_C": zeta_deg,
        },
        "combination": combo_deg,
        "P": P_deg.tolist(),
        "max_a_error_vs_GR": max_a_deg_err,
        "max_shape_error_vs_GR": max_shape_deg_err,
        "max_sigma2_error_vs_GR": max_sigma_deg_err,
    },
    "locks": {
        "generic_full_rate_match_onshell":
            G2831_GENERIC_FULL_RATE_MATCH_ONSHELL,
        "couplings_numerically_derived":
            G2831_COUPLINGS_NUMERICALLY_DERIVED,
        "deviation_uniquely_cubic_attributable":
            G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE,
        "inhomogeneous_field_derived":
            G2831_INHOMOGENEOUS_FIELD_DERIVED,
        "Mercury_GVH_precession_prediction_authorized":
            G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED,
        "new_GVH_physics_validated":
            G2831_NEW_GVH_PHYSICS_VALIDATED,
        "real_data_ready":
            G2831_REAL_DATA_READY,
    },
    "verdict": {
        "G2831_LOCAL_AUDIT_PASS": G2831_LOCAL_AUDIT_PASS,
        "obstructions": G2831_OBSTRUCTIONS,
        "next_authorized": G2831_NEXT_AUTHORIZED,
    },
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.28.3.1_"
    "GR_vs_GVH_DCC_Matched_Initial_Shape_Evolution_Audit_FAST.json"
)
artifact_path.write_text(
    json.dumps(artifact, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("LEVEL1 =", LEVEL1)
print("LEVEL2 =", LEVEL2)
print("LEVEL3 =", LEVEL3)
print("LEVEL4 =", LEVEL4)
print("G2831_GR_VS_GVH_MATCHED_SHAPE_GATE_CLOSED =",
      G2831_GR_VS_GVH_MATCHED_SHAPE_GATE_CLOSED)
print("G2831_NUMERIC_BENCHMARK_GATE_CLOSED =",
      G2831_NUMERIC_BENCHMARK_GATE_CLOSED)
print("G2831_PARAMETERIZED_SHAPE_DEVIATION_MATERIALIZED =",
      G2831_PARAMETERIZED_SHAPE_DEVIATION_MATERIALIZED)
print("G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE =",
      G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE)
print("G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED =",
      G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED)
print("G2831_NEW_GVH_PHYSICS_VALIDATED =", G2831_NEW_GVH_PHYSICS_VALIDATED)
print("G2831_LOCAL_AUDIT_PASS =", G2831_LOCAL_AUDIT_PASS)
print("G2831_OBSTRUCTIONS =", G2831_OBSTRUCTIONS)
print("G2831_NEXT_AUTHORIZED =", G2831_NEXT_AUTHORIZED)
print("G2831 artifact =", artifact_path)

LEVEL1 = GR_LIMIT_AND_MATCHED_INITIAL_GEOMETRY_PROTOCOL_EXACTLY_CLASSIFIED
LEVEL2 = PARAMETERIZED_SHAPE_SHEAR_AND_AXIS_EVOLUTION_DIFFERENCES_MATERIALIZED
LEVEL3 = NUMERIC_NONDEGENERATE_AND_NONZERO_COUPLING_GR_DEGENERACY_WITNESSES_PASS
LEVEL4 = BLOCKED_UNIQUE_CUBIC_ATTRIBUTION_INHOMOGENEOUS_FIELD_AND_MERCURY_PREDICTION
G2831_GR_VS_GVH_MATCHED_SHAPE_GATE_CLOSED = True
G2831_NUMERIC_BENCHMARK_GATE_CLOSED = True
G2831_PARAMETERIZED_SHAPE_DEVIATION_MATERIALIZED = True
G2831_DEVIATION_UNIQUELY_CUBIC_ATTRIBUTABLE = False
G2831_MERCURY_GVH_PRECESSION_PREDICTION_AUTHORIZED = False
G2831_NEW_GVH_PHYSICS_VALIDATED = False
G2831_LOCAL_AUDIT_PASS = True
G2831_OBSTRUCTIONS = ['MATCHED_FULL_INITIAL_RATES_ARE_GENERICALLY_OFFSHELL_BETWEEN_THE_TWO_THEORIES', 'COUPLINGS_c13_c2_zetaC_ARE_NOT_NUMERICALLY_THEORY_SELECTED_HERE', 'ALIGNED_BRANCH_DEVIATION_IS_NOT_UNIQUELY_ATTRIBUTABLE_TO_THE_CUBIC_OPERATOR', 'GENERAL_INHOMOGENEOUS_D_phys_OF_t_x_NOT_DERIVED', 'LOCALIZED_SOURCE_AND_NBODY_SPATIAL_MATCHING_NOT_MAT